In [1]:
#Read all Gaussian fit log files from a given directory and convert them to an appropriate csv file

import os
import glob
import csv
import math

In [2]:
#simple auxiliary functions to convert channels to km/s given a range of channels

def chanToKmS(value,nChans,vel0,velN):
    return vel0 + value * ((velN - vel0)/nChans)

def chanToKmSE(value,nChans,vel0,velN):
    return value * abs((velN - vel0)/nChans)

In [4]:
# To-do: improve so this is less hard-coded
# This block extracts and calculates values of interest for every file in a given directory

dirpath = "./results/gaussianFits/wholeRegions/"

longLat = False
longLatFK5 = True
scaleFactor = False

for filename in glob.glob(os.path.join(dirpath, '*Stats*')):
    #file by file
    with open(os.path.join(os.getcwd(), filename), 'r') as f:
        #line by line
        lines = (f.readlines())
        sum, mean = 0, 0
        for l in lines:
            if longLat:
                if "wcs:FK5" in l:
                    # print(l.split())
                    # print(l.split()[1][22:-1])
                    # print(filename)
                    long = (int(l.split()[1][17:18])*15) + (float(l.split()[1][19:21])/4) + (float(l.split()[1][22:-1])/240)
                    lat = (int(l.split()[2][0:2])) + (float(l.split()[2][3:5])/60) + (float(l.split()[2][6:-2])/3600)
                    print(long,"\t",lat)
            if longLatFK5:
                if "wcs:FK5" in l:
                    long = (l.split()[1][17:26])
                    lat = (l.split()[2][0:10])
                    print(long,"\t",lat)
            if scaleFactor:
                if "Sum" in l:
                    sum = float(l.split()[1])
                elif "Mean" in l:
                    mean = float(l.split()[1])
                    print(sum)
                    print(mean)
                    scaleFactor = sum/mean
                    print(scaleFactor)

1:24:39.1 	 37:21:20.8
1:03:16.7 	 40:32:18.0
1:06:18.2 	 39:23:41.9
1:03:55.6 	 39:29:30.2
0:59:30.1 	 39:31:05.2
1:03:01.0 	 38:34:05.7
1:03:06.1 	 39:05:13.7
0:59:06.0 	 38:52:33.8
0:48:27.6 	 38:52:22.5
0:52:24.3 	 39:15:31.5
1:21:09.4 	 37:17:02.4
1:19:28.9 	 37:26:20.9
1:08:52.6 	 37:38:53.6
1:05:06.9 	 36:22:53.6
1:03:15.2 	 35:59:27.4
1:17:39.7 	 36:42:21.2
0:51:41.5 	 39:50:08.8
0:43:58.6 	 39:31:21.0
1:24:35.1 	 38:19:29.6
0:58:39.5 	 38:05:08.2
1:01:17.4 	 36:13:25.7
1:15:16.3 	 34:59:37.6


In [30]:
#ideally fit should already be done in km/s (or MHz), but if channel information is known, change these values
#before running, ensure that all log files for a given source are in the same directory, and that directory paths are correct

source = "regionH"
dirpath = "./results/gaussianFits/" + source
outputpath = "./results/cloudParameters/" + source + ".csv"

# nChans = 621
# vel0 = 400.263
# velN = -399.7536
nChans = 1244
vel0 = 400.585
velN = -400.7198

#make sure we don't duplicate pixels - floor pixels and check
#convert coordinates to actual numbers

pixels = []
unit = ""
csvData = [["Pixel", "Long (degrees)", "Lat (degrees)", "amp (K)", "amp e (K)", "center (km/s)", "center e (km/s)", "FWHM (km/s)", "FWHM e (km/s)", "integral (K*km/s)", "integral e (K*km/s)"]]

#extract information from log files
#to-do (if time allows): this could be less hard-coded
for filename in glob.glob(os.path.join(dirpath, '*.txt')):
    #file by file
    with open(os.path.join(os.getcwd(), filename), 'r') as f:
        #line by line
        lines = (f.readlines())
        skip = False
        isPoint = False
        for l in lines:
            if "pixel" in l:
                isPoint = True
                pix1 = math.floor(float(l.split()[3][1:10]))
                pix2 = math.floor(float(l.split()[4][0:9]))
                pCurrent = (pix1,pix2)
                # print(pCurrent)
                for p in pixels:
                    if pCurrent == p:
                        skip = True
                if not skip:
                    pixels.append((pix1,pix2))
            if "wcs:FK5" in l:
                long = (int(l.split()[3][1:2])*15) + (float(l.split()[3][3:5])/4) + (float(l.split()[3][6:-2])/240)
                lat = (int(l.split()[4][0:2])) + (float(l.split()[3][3:5])/60) + (float(l.split()[3][6:-2])/3600)
            if "amp1" in l:
                amp = float(l.split()[2])
                ampE = float(l.split()[5])
            if "center1" in l:
                center = float(l.split()[2])
                centerE = float(l.split()[5])
                if "km/s" in l.split()[3]:
                    unit = "km/s"
                elif "Channel" in l.split()[3]:
                    unit = "Channel"
                    center = chanToKmS(center, nChans, vel0, velN)
                    centerE = chanToKmSE(centerE, nChans, vel0, velN)
            if "fwhm1" in l:
                fwhm = float(l.split()[2])
                fwhmE = float(l.split()[5])
                if unit == "Channel":
                    fwhm = chanToKmSE(fwhm, nChans, vel0, velN)
                    fwhmE = chanToKmSE(fwhmE, nChans, vel0, velN)
            if "integral" in l:
                integral = float(l.split()[4])
                integralE = float(l.split()[9])
                if unit == "Channel":
                    integral = chanToKmSE(integral, nChans, vel0, velN)
                    integralE = chanToKmSE(integralE, nChans, vel0, velN)
        if (not skip) and isPoint:
            # print("appending")
            csvData.append([pCurrent,long,lat,amp,ampE,center,centerE,fwhm,fwhmE,integral,integralE])

#write to CSV
with open(outputpath, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerows(csvData)

print(outputpath)

./results/cloudParameters/regionH.csv
